# 03 — Anomaly Detection in Sensor Events

This notebook builds an unsupervised anomaly detection workflow for operational sensor data.

The hidden anomaly labels are used only after scoring, as an offline evaluation tool.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.covariance import EllipticEnvelope
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

sys.path.append(str(Path.cwd() / "src"))

from unsup_lab.data import make_sensor_anomaly_data


In [2]:
dataset = make_sensor_anomaly_data(n_points=3_000, contamination=0.04, random_state=42)
raw = dataset.features
labels = dataset.hidden_labels

raw.head()


,timestamp,temperature,motion_count,power_usage,signal_strength,missing_ratio
0,2026-01-01 00:00:00,21.182830,8.0,3.639946,-67.584958,0.049494
1,2026-01-01 00:15:00,20.376010,2.0,0.822426,-61.593665,0.100338
2,2026-01-01 00:30:00,21.450271,4.0,0.987560,-59.467281,0.063276
3,2026-01-01 00:45:00,21.564339,7.0,2.777653,-60.135436,0.216096
4,2026-01-01 01:00:00,20.476426,2.0,1.728149,-64.278383,0.002398


In [3]:
feature_columns = [
    "temperature",
    "motion_count",
    "power_usage",
    "signal_strength",
    "missing_ratio",
]

x = StandardScaler().fit_transform(raw[feature_columns])
y_true = (labels == "anomaly").astype(int).to_numpy()


## Score multiple anomaly detectors

In [4]:
scores = pd.DataFrame(index=raw.index)

isolation = IsolationForest(contamination=0.04, random_state=42)
isolation.fit(x)
scores["isolation_forest"] = -isolation.score_samples(x)

lof = LocalOutlierFactor(n_neighbors=35, contamination=0.04)
lof_labels = lof.fit_predict(x)
scores["local_outlier_factor"] = -lof.negative_outlier_factor_

robust_cov = EllipticEnvelope(contamination=0.04, random_state=42)
robust_cov.fit(x)
scores["robust_covariance"] = -robust_cov.score_samples(x)

svm = OneClassSVM(nu=0.04, gamma="scale")
svm.fit(x)
scores["one_class_svm"] = -svm.score_samples(x)

pca = PCA(n_components=2, random_state=42)
x_low = pca.fit_transform(x)
x_reconstructed = pca.inverse_transform(x_low)
scores["pca_reconstruction"] = np.mean((x - x_reconstructed) ** 2, axis=1)

scores.head()


,isolation_forest,local_outlier_factor,robust_covariance,one_class_svm,pca_reconstruction
0,0.444858,1.202720,7.186113,-11.623714,0.172471
1,0.445797,1.036837,4.090682,-12.712706,0.110287
2,0.448113,1.124723,6.726786,-12.164813,0.203478
3,0.470969,1.196191,14.195591,-11.971130,0.310426
4,0.414441,1.033800,4.982857,-11.536622,0.091086


## Precision at top-k

In [5]:
def precision_at_k(score_values, y_true, k):
    top_indices = np.argsort(score_values)[-k:]
    return y_true[top_indices].mean()

k = int(y_true.sum())
results = {
    column: precision_at_k(scores[column].to_numpy(), y_true, k)
    for column in scores.columns
}

pd.Series(results, name=f"precision_at_{k}").sort_values(ascending=False)


robust_covariance       1.000000
isolation_forest        0.991667
one_class_svm           0.508333
pca_reconstruction      0.391667
local_outlier_factor    0.050000
Name: precision_at_120, dtype: float64

## Combining the detectors, and reviewing the top anomalies

No single detector is trustworthy on its own, so the tempting move is to average them. But the five scores are not on a common scale — robust covariance is a Mahalanobis-style distance spanning hundreds, while the isolation forest lives in tenths — so a plain mean is really just the largest-scale detector wearing a disguise. Ranking each detector first puts them on the same `[0, 1]` footing before they vote.

In [6]:
raw_mean = scores.mean(axis=1)
ensemble_score = scores.rank(pct=True).mean(axis=1)

# How much of a plain mean is each detector actually responsible for?
print("correlation with a plain mean of the raw scores:")
print(scores.corrwith(raw_mean).round(3).to_string())
print(
    f"\nprecision@{k}: plain mean {precision_at_k(raw_mean.to_numpy(), y_true, k):.3f}"
    f"  |  rank ensemble {precision_at_k(ensemble_score.to_numpy(), y_true, k):.3f}"
)

review = raw.copy()
review["anomaly_score"] = ensemble_score
review["hidden_label_for_offline_eval"] = labels

review.sort_values("anomaly_score", ascending=False).head(10)

correlation with a plain mean of the raw scores:
isolation_forest        0.788
local_outlier_factor    0.036
robust_covariance       1.000
one_class_svm           0.429
pca_reconstruction      0.452

precision@120: plain mean 1.000  |  rank ensemble 0.242


,timestamp,temperature,motion_count,power_usage,signal_strength,missing_ratio,anomaly_score,hidden_label_for_offline_eval
1848,2026-01-20 06:00:00,33.041119,10.0,8.376297,-74.869518,0.936146,0.988000,anomaly
525,2026-01-06 11:15:00,25.957102,10.0,10.067112,-79.780251,0.453584,0.986600,anomaly
2515,2026-01-27 04:45:00,30.251906,7.0,10.413447,-85.000085,0.468342,0.982800,anomaly
1301,2026-01-14 13:15:00,24.565348,1.0,6.013243,-96.867316,0.526412,0.981800,anomaly
1715,2026-01-18 20:45:00,26.295109,2.0,5.108769,-96.606326,0.885671,0.981200,anomaly
2840,2026-01-30 14:00:00,27.526956,7.0,8.305380,-67.284168,0.455032,0.979733,anomaly
1994,2026-01-21 18:30:00,18.552721,8.0,3.271701,-67.700867,0.341893,0.978000,normal
421,2026-01-05 09:15:00,32.684632,10.0,9.752822,-85.050670,0.512782,0.976533,anomaly
1071,2026-01-12 03:45:00,22.098997,12.0,3.634457,-53.335190,0.179086,0.974133,normal
16,2026-01-01 04:00:00,30.229209,7.0,9.692563,-73.555009,0.747847,0.974133,anomaly


## Interpretation

Two honest readings come out of the numbers above.

**The naive ensemble was an illusion.** Averaging the raw scores correlates ~1.00 with robust covariance alone: its scale simply drowns the other four votes. The high precision that mean achieves is robust covariance's precision, not the ensemble's — a result that would collapse the moment a differently-scaled detector joined the pool.

**Once the detectors vote on equal terms, they disagree.** The rank ensemble scores well below the best individual detector, because local outlier factor performs near chance here (precision 0.05) and drags four honest votes down with it. Averaging detectors is not a free upgrade: it pays off when the members are individually decent and make *different* mistakes, and it costs when one member is systematically wrong.

The practical read for this dataset: the injected anomalies are global, multivariate outliers, which is exactly what a Mahalanobis-style detector is built for and exactly what a local-density method like LOF is not. Match the detector to the anomaly you expect, and rank the ensemble members before trusting a combination.

## Limitations

Anomaly scores are not probabilities. Thresholds should be calibrated with analyst feedback, operational cost, and tolerance for false positives.
